In [26]:
from langchain_ollama import OllamaLLM

model = OllamaLLM(model="llama3.2:latest")


In [27]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [28]:
chain = model | parser

In [29]:
from langchain.prompts import ChatPromptTemplate

template = """
Answer the question based on the context below. If you can't 
answer the question, reply "I don't know".

Context: {context}

Question: {question}
"""
 
prompt = ChatPromptTemplate.from_template(template)


In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("R22_syllabus.pdf")

In [6]:
pages = loader.load_and_split()

In [7]:
from langchain_community.embeddings import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="bge-m3")

/tmp/ipykernel_2739/3619592484.py:3: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="bge-m3")


In [16]:
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents=pages, embedding=embeddings)

In [25]:
retriever = vectorstore.as_retriever(
    # search_kwargs={'k': 5}
)
retriever.invoke("list all the Courses in 1 year Semester 1")

[Document(metadata={'page': 8, 'page_label': '9', 'source': 'R22_syllabus.pdf'}, page_content='B.\n \nTech\n \nComputer\n \nScience and\n \nEngineering\n \nwith AI\n \n&\n \nML\n \n(R-22\n \nRegulation)\n \n \nI\n \nYear\n \n–\n \nI\n \nSemester\n \n \n \n \nCourse  \ncode \n \nCategory \n \nCourse Title \nHours per \nweek \n \nInternal \nMarks \n \nExternal \nMarks \n \nTotal \nMarks \n \nCredits \nL T P  \n \nCSM1101 \n \nBS \nEngineering Mathematics-I (Partial \nDifferentiation, Multiple Integrals, \nFourier Series and Applications) \n \n3 \n \n0 \n \n0 \n \n30 \n \n70 \n \n100 \n \n3 \nCSM1102 BS Green Chemistry \n3 \n1 \n0 30 70 100 \n3 \nCSM1103 HSS English \n3 \n0 \n0 30 70 100 \n3 \n \nCSM1104 \n \nES Computer Programming Using ‘C’  \n3 \n \n0 \n \n0 \n \n30 \n \n70 \n \n100 \n \n3 \n \nCSM1105 \n \nES \n \nIT Essentials \n \n3 \n \n0 \n \n0 \n \n30 \n \n70 \n \n100 \n3 \n \nCSM1106 \n \nHSS \n \nCommunication Skills Lab \n \n0 \n \n0 \n \n3 \n \n50 \n \n50 \n \n \n100 \n \n1.5

In [30]:
from operator import itemgetter

chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
    }
    | prompt
    | model
    | parser
)

In [31]:
questions = [
    "list all the Courses in Semester 1",
]

for question in questions:
    print(f"Question: {question}")
    print(f"Answer: {chain.invoke({'question': question})}")
    print()

Question: list all the Courses in Semester 1
Answer: Here are the courses listed for Semester 1:

1. CSM1101 - BS Engineering Mathematics-I (Partial Differentiation, Multiple Integrals, Fourier Series and Applications)
2. CSM1102 - Green Chemistry
3. HSS English
4. CSM1104 - ES Computer Programming Using 'C'
5. CSM1105 - IT Essentials

